# AML/TM PySpark DataFrame Basics Notebook

Run this notebook top to bottom in Azure Databricks, Fabric Spark notebooks, or a Jupyter environment with PySpark. It creates tiny AML/TM training data, performs DataFrame transformations, validates expected output, and produces one explainable alert.

## Step 0 - Bootstrap

Expected setup: `transactions_raw` has 8 rows, `accounts` has 4 rows, and `country_risk` has 3 rows.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("aml-notebook-pyspark-basics").getOrCreate()

def assert_set(name, actual_rows, expected_rows):
    actual = set(actual_rows)
    expected = set(expected_rows)
    assert actual == expected, f"{name}: expected {expected}, got {actual}"

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("account_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("amount_cad", T.StringType(), False),
    T.StructField("transaction_type", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("country_code", T.StringType(), True),
])

transactions_raw = spark.createDataFrame([
    ("t1", "a1", "2022-06-01", "60.00", "WIRE", "POSTED", "IR"),
    ("t2", "a1", "2022-06-03", "50.00", "WIRE", "POSTED", "IR"),
    ("t3", "a1", "2022-06-05", "10.00", "CARD", "POSTED", "CA"),
    ("t4", "a2", "2022-06-02", "200.00", "WIRE", "POSTED", "CA"),
    ("t5", "a3", "2022-06-02", "20.00", "WIRE", "REVERSED", "IR"),
    ("t6", "a9", "2022-06-02", "80.00", "WIRE", "POSTED", "IR"),
    ("t7", "a2", "2022-07-01", "300.00", "WIRE", "POSTED", "IR"),
    ("t8", "a4", "2022-06-10", "100.00", "CASH", "POSTED", None),
], schema=transaction_schema)

accounts = spark.createDataFrame([
    ("a1", "c1", "ACTIVE", "CHECKING"),
    ("a2", "c2", "ACTIVE", "CHECKING"),
    ("a3", "c3", "ACTIVE", "SAVINGS"),
    ("a4", "c4", "CLOSED", "CHECKING"),
], ["account_id", "customer_id", "account_status", "product_type"])

country_risk = spark.createDataFrame([
    ("IR", "HIGH"),
    ("CA", "LOW"),
    ("US", "LOW"),
], ["country_code", "risk_level"])

assert transactions_raw.count() == 8
assert accounts.count() == 4
assert country_risk.count() == 3
print("Bootstrap validation passed.")

## Step 1 - Normalize Types

Spark reads the raw date and amount as strings here. Cast them before doing date filters or numeric thresholds.

In [ ]:
transactions = (
    transactions_raw
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
    .withColumn("amount_cad", F.col("amount_cad").cast("decimal(18,2)"))
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("country_code", F.upper(F.trim("country_code")))
)

assert transactions.count() == 8
assert dict(transactions.dtypes)["transaction_date"] == "date"
assert dict(transactions.dtypes)["amount_cad"] == "decimal(18,2)"
transactions.orderBy("transaction_id").show(truncate=False)

## Step 2 - Filter Basics

Predict the row ids before running each filter. The important null lesson: `country_code != 'CA'` does not return null country rows.

In [ ]:
posted_wires = transactions.filter(F.col("status") == "POSTED").filter(F.col("transaction_type") == "WIRE")
june_transactions = transactions.filter(
    (F.col("transaction_date") >= F.lit("2022-06-01"))
    & (F.col("transaction_date") < F.lit("2022-07-01"))
)
missing_country = transactions.filter(F.col("country_code").isNull())
not_ca = transactions.filter(F.col("country_code") != "CA")
not_ca_or_missing = transactions.filter((F.col("country_code") != "CA") | F.col("country_code").isNull())

assert posted_wires.count() == 5
assert june_transactions.count() == 7
assert_set("missing_country", [r.transaction_id for r in missing_country.select("transaction_id").collect()], ["t8"])
assert "t8" not in {r.transaction_id for r in not_ca.select("transaction_id").collect()}
assert "t8" in {r.transaction_id for r in not_ca_or_missing.select("transaction_id").collect()}

posted_wires.orderBy("transaction_id").show(truncate=False)

## Step 3 - Joins, DQ, and Row Loss

The inner join hides orphan transaction `t6`. The left anti join makes the DQ exception explicit.

In [ ]:
tx_with_accounts_inner = transactions.join(accounts, on="account_id", how="inner")
tx_with_accounts_left = transactions.join(accounts, on="account_id", how="left")
orphan_accounts = transactions.join(accounts, on="account_id", how="left_anti")
valid_account_transactions = transactions.join(accounts, on="account_id", how="left_semi")
tx_with_risk = transactions.join(country_risk, on="country_code", how="left")
high_risk_tx = tx_with_risk.filter(F.col("risk_level") == "HIGH")

assert tx_with_accounts_inner.count() == 7
assert tx_with_accounts_left.count() == 8
assert_set("orphan_accounts", [r.transaction_id for r in orphan_accounts.select("transaction_id").collect()], ["t6"])
assert valid_account_transactions.count() == 7
assert_set("high_risk_tx", [r.transaction_id for r in high_risk_tx.select("transaction_id").collect()], ["t1", "t2", "t5", "t6", "t7"])

orphan_accounts.show(truncate=False)

## Step 4 - Window Function

`row_number` lets you pick the latest transaction per account using a deterministic tie-breaker.

In [ ]:
latest_window = Window.partitionBy("account_id").orderBy(
    F.col("transaction_date").desc(),
    F.col("transaction_id").desc(),
)
latest_by_account = transactions.withColumn("rn", F.row_number().over(latest_window)).filter(F.col("rn") == 1).drop("rn")

latest_pairs = {(r.account_id, r.transaction_id) for r in latest_by_account.select("account_id", "transaction_id").collect()}
assert latest_pairs == {("a1", "t3"), ("a2", "t7"), ("a3", "t5"), ("a4", "t8"), ("a9", "t6")}
latest_by_account.orderBy("account_id").show(truncate=False)

## Step 5 - Build an Explainable AML/TM Alert

Rule idea: for June 2022, find posted WIRE transactions from high-risk countries, join to valid accounts, group by customer, and alert when total amount is greater than 100 CAD.

In [ ]:
june_posted_wires = (
    transactions.filter(F.col("status") == "POSTED")
    .filter(F.col("transaction_type") == "WIRE")
    .filter((F.col("transaction_date") >= F.lit("2022-06-01")) & (F.col("transaction_date") < F.lit("2022-07-01")))
)
valid_customer_tx = june_posted_wires.join(accounts, on="account_id", how="inner")
high_risk_customer_tx = valid_customer_tx.join(country_risk, on="country_code", how="inner").filter(F.col("risk_level") == "HIGH")
customer_totals = high_risk_customer_tx.groupBy("customer_id").agg(
    F.sum("amount_cad").alias("observed_amount_cad"),
    F.count("*").alias("supporting_transaction_count"),
)
alerts = (
    customer_totals.filter(F.col("observed_amount_cad") > F.lit(100))
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
    .withColumn("processing_month", F.lit("2022-06"))
    .withColumn("alert_key", F.sha2(F.concat_ws("|", "rule_id", "rule_version", "processing_month", "customer_id"), 256))
)
supporting_transactions = high_risk_customer_tx.join(alerts.select("alert_key", "customer_id"), on="customer_id", how="inner")

assert alerts.count() == 1
assert_set("alert customers", [r.customer_id for r in alerts.select("customer_id").collect()], ["c1"])
assert_set("supporting transaction ids", [r.transaction_id for r in supporting_transactions.select("transaction_id").collect()], ["t1", "t2"])

alerts.select("alert_key", "customer_id", "observed_amount_cad", "supporting_transaction_count").show(truncate=False)
supporting_transactions.select("alert_key", "transaction_id", "customer_id", "amount_cad", "country_code", "risk_level").orderBy("transaction_id").show(truncate=False)

## Step 6 - Reconciliation

A learning notebook should end with evidence. These counts explain where rows moved, where rows dropped, and why one alert was generated.

In [ ]:
reconciliation = spark.createDataFrame([
    ("transactions", transactions.count()),
    ("posted_wires", posted_wires.count()),
    ("june_posted_wires", june_posted_wires.count()),
    ("valid_customer_tx", valid_customer_tx.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("high_risk_customer_tx", high_risk_customer_tx.count()),
    ("alerts", alerts.count()),
], ["step_name", "row_count"])

expected_counts = {
    "transactions": 8,
    "posted_wires": 5,
    "june_posted_wires": 4,
    "valid_customer_tx": 3,
    "orphan_accounts": 1,
    "high_risk_customer_tx": 2,
    "alerts": 1,
}
actual_counts = {r.step_name: r.row_count for r in reconciliation.collect()}
assert actual_counts == expected_counts, f"Expected {expected_counts}, got {actual_counts}"
reconciliation.show(truncate=False)
print("Notebook validation passed.")

## Closed-Book Drill

Before rerunning, change `t2` from `IR` to `CA`. Predict the new `high_risk_customer_tx`, `customer_totals`, and `alerts` counts.